In [1]:
import sys
from pathlib import Path

# Add notebooks root to path for utils import
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from utils import load_aggregate_results

all_results = load_aggregate_results(category="test-generation")
print(f"Loaded results for {len(all_results)} setups")

Loaded results for 2 setups


In [2]:
from typing import Any

import numpy as np
import plotly.graph_objects as go

from bcbench.results.metrics import pass_at_k, pass_hat_k


def bootstrap_ci(data: np.ndarray, n_bootstrap: int = 10000, ci_level: float = 0.95) -> dict[str, Any]:
    """Compute bootstrap confidence interval using percentile method."""
    rng = np.random.default_rng(42)
    bootstrap_means = np.array([rng.choice(data, size=len(data), replace=True).mean() for _ in range(n_bootstrap)])
    alpha = 1 - ci_level
    ci_low = np.percentile(bootstrap_means, alpha / 2 * 100)
    ci_high = np.percentile(bootstrap_means, (1 - alpha / 2) * 100)
    return {
        "mean": data.mean(),
        "ci_low": ci_low,
        "ci_high": ci_high,
        "ci_half": (ci_high - ci_low) / 2,
        "bootstrap_means": bootstrap_means,
    }

## 1. Mean Resolution Rate with Bootstrap 95% CI

Compare the mean resolution rates between the two setups with confidence intervals.

In [3]:
# Calculate mean resolution rate with bootstrap CI for each setup
ci_data = []
for setup_name, setup_df in all_results.items():
    per_run_means = setup_df.groupby("run_id")["resolved"].mean().values * 100  # type: ignore
    n_runs = len(per_run_means)

    if n_runs >= 2:
        boot = bootstrap_ci(np.array(per_run_means))
        ci_data.append(
            {
                "Setup": setup_name,
                "N Runs": n_runs,
                "Mean %": boot["mean"],
                "CI Low": boot["ci_low"],
                "CI High": boot["ci_high"],
                "CI ±": boot["ci_half"],
            }
        )

ci_df = pd.DataFrame(ci_data).sort_values("Mean %", ascending=True)

# Chart: Mean resolution rate with CI
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=ci_df["Mean %"],
        y=ci_df["Setup"],
        mode="markers",
        marker={"size": 14, "color": ["steelblue", "coral"][: len(ci_df)]},
        error_x={
            "type": "data",
            "symmetric": False,
            "array": ci_df["CI High"] - ci_df["Mean %"],
            "arrayminus": ci_df["Mean %"] - ci_df["CI Low"],
            "thickness": 2,
            "width": 10,
        },
        name="Mean ± 95% CI",
    )
)

x_min = max(0, ci_df["CI Low"].min() - 5)
x_max = min(100, ci_df["CI High"].max() + 5)

fig.update_layout(
    title="Mean Resolution Rate with 95% Bootstrap CI",
    xaxis_title="Resolution Rate (%)",
    xaxis={"range": [x_min, x_max], "dtick": 5, "showgrid": True, "gridcolor": "lightgray"},
    yaxis_title="",
    height=300,
    showlegend=False,
    margin={"l": 200},
)
fig.show()

# Table
table_df = ci_df.copy()
table_df["Mean ± 95% CI"] = table_df.apply(lambda r: f"{r['Mean %']:.1f}% ± {r['CI ±']:.1f}%", axis=1)
table_df[["Setup", "N Runs", "Mean ± 95% CI"]]

,Setup,N Runs,Mean ± 95% CI
0,altest-opus-4-5,5,38.6% ± 2.8%
1,default-opus-4-5,5,45.5% ± 2.5%


## 2. pass@k Comparison

**pass@k** = 1 - C(n-c,k)/C(n,k) — probability of at least 1 success when sampling k runs.

In [4]:
# Calculate pass@k for different values of k
def compute_pass_at_k_for_k_values(df: pd.DataFrame, max_k: int) -> list[float]:
    """Compute pass@k for k=1 to max_k."""
    run_ids_sorted = sorted(df["run_id"].unique())
    pivot = df.pivot_table(index="instance_id", columns="run_id", values="resolved", aggfunc=lambda x: x.iloc[0])
    pivot = pivot[run_ids_sorted]
    n_instances = len(pivot)

    pass_at_k_pcts = []
    for k in range(1, max_k + 1):
        first_k_runs = run_ids_sorted[:k]
        pivot_k = pivot[first_k_runs]
        total = sum(pass_at_k(k, int(row.sum()), k) for _, row in pivot_k.iterrows())
        pass_at_k_pcts.append(total / n_instances * 100)
    return pass_at_k_pcts


# Determine max k (minimum runs across setups)
max_k = min(df["run_id"].nunique() for df in all_results.values())
k_values = list(range(1, max_k + 1))

# Calculate pass@k for each setup
pass_at_k_data = {}
for setup_name, setup_df in all_results.items():
    pass_at_k_data[setup_name] = compute_pass_at_k_for_k_values(setup_df, max_k)

# Chart: pass@k comparison
colors = {"default-opus-4-5": "coral", "altest-opus-4-5": "steelblue"}
fig = go.Figure()

for setup_name, pass_at_k_values in pass_at_k_data.items():
    fig.add_trace(
        go.Scatter(
            x=k_values,
            y=pass_at_k_values,
            mode="lines+markers",
            name=setup_name,
            line={"color": colors.get(setup_name, "gray")},
            marker={"size": 8},
        )
    )

fig.update_layout(
    title="pass@k Comparison: Probability of ≥1 Success in k Runs",
    xaxis_title="k (number of runs)",
    yaxis_title="pass@k (%)",
    yaxis={"range": [0, 100]},
    xaxis={"dtick": 1},
    height=400,
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "center", "x": 0.5},
)
fig.show()

# Table: pass@k values
pass_at_k_table = pd.DataFrame({"k": k_values})
for setup_name, values in pass_at_k_data.items():
    pass_at_k_table[f"{setup_name} pass@k %"] = [f"{v:.1f}" for v in values]
pass_at_k_table

,k,altest-opus-4-5 pass@k %,default-opus-4-5 pass@k %
0,1,33.7,43.6
1,2,51.5,55.4
2,3,63.4,61.4
3,4,65.3,63.4
4,5,69.3,68.3


## 3. pass^k Comparison

**pass^k** = C(c,k)/C(n,k) — probability that ALL k sampled runs succeed.

In [5]:
# Calculate pass^k for different values of k
def compute_pass_hat_k_for_k_values(df: pd.DataFrame, max_k: int) -> list[float]:
    """Compute pass^k for k=1 to max_k."""
    run_ids_sorted = sorted(df["run_id"].unique())
    pivot = df.pivot_table(index="instance_id", columns="run_id", values="resolved", aggfunc=lambda x: x.iloc[0])
    pivot = pivot[run_ids_sorted]
    n_instances = len(pivot)

    pass_hat_k_pcts = []
    for k in range(1, max_k + 1):
        first_k_runs = run_ids_sorted[:k]
        pivot_k = pivot[first_k_runs]
        total = sum(pass_hat_k(k, int(row.sum()), k) for _, row in pivot_k.iterrows())
        pass_hat_k_pcts.append(total / n_instances * 100)
    return pass_hat_k_pcts


# Calculate pass^k for each setup
pass_hat_k_data = {}
for setup_name, setup_df in all_results.items():
    pass_hat_k_data[setup_name] = compute_pass_hat_k_for_k_values(setup_df, max_k)

# Chart: pass^k comparison
fig = go.Figure()

for setup_name, pass_hat_k_values in pass_hat_k_data.items():
    fig.add_trace(
        go.Scatter(
            x=k_values,
            y=pass_hat_k_values,
            mode="lines+markers",
            name=setup_name,
            line={"color": colors.get(setup_name, "gray")},
            marker={"size": 8},
        )
    )

fig.update_layout(
    title="pass^k Comparison: Probability ALL k Runs Succeed",
    xaxis_title="k (number of runs)",
    yaxis_title="pass^k (%)",
    yaxis={"range": [0, 100]},
    xaxis={"dtick": 1},
    height=400,
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "center", "x": 0.5},
)
fig.show()

# Table: pass^k values
pass_hat_k_table = pd.DataFrame({"k": k_values})
for setup_name, values in pass_hat_k_data.items():
    pass_hat_k_table[f"{setup_name} pass^k %"] = [f"{v:.1f}" for v in values]
pass_hat_k_table

,k,altest-opus-4-5 pass^k %,default-opus-4-5 pass^k %
0,1,33.7,43.6
1,2,21.8,31.7
2,3,13.9,23.8
3,4,11.9,21.8
4,5,11.9,20.8
